# A1.3 · Architecture review when the system acts

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.2 · The controls, and where each one binds](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**.

| | |
|---|---|
| Open-source tooling | OWASP Threat Dragon, kagent |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Start with something that is not an agent.

A **model** takes text and returns text. Ask GPT-4 or Llama 3.3 to delete your
production database and it will produce a convincing paragraph about deleting
your production database. Nothing happens. The output is a string.

A model becomes an **agent** when something reads that string and *acts on it* —
a program that sees `{"tool": "run_sql", "args": {...}}` and actually connects
to the database. That program is the agent. The model is a component inside it.

This gives us three layers, and the whole curriculum uses these names:

| Plane | What lives here | Can it change the world? |
|---|---|---|
| **Decision** | the model. Proposals, plans, text. | No. Never. |
| **Control** | policy, scopes, approval gates, the gateway. | It decides what passes. |
| **Action** | the tools. SQL, HTTP, the filesystem, the cloud API. | Yes. Only here. |

The consequence is the point of this lesson. When something goes wrong, "the
model did it" cannot be the root cause — the model only ever wrote on the
decision plane. Something on the **control plane** let a proposal through. That
is where your architecture review has to look.

Traditional architecture review assumed the system's behaviour was fixed at
design time: you read the code, you drew the trust boundaries, you signed the
document. An agent's behaviour is determined by a *tool manifest* that changes
when someone edits a config file. So we need a review that runs against the
manifest, continuously — which means the manifest has to be something a program
can read.

## 2 · Demo — classify a real tool manifest

Here is the manifest of a plausible internal agent: a bot that triages security findings. Every entry is a capability someone actually grants in real deployments. The question the code answers is which plane each one sits on — and note that the answer comes from what a tool *can do*, never from what it is called.

In [ ]:
from dataclasses import dataclass, field

# How far a single call can reach. These weights are the crude, useful kind:
# the absolute number means nothing, the *ratio* between designs means a lot.
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20, "internet": 50}

@dataclass(frozen=True)
class Tool:
    name: str
    writes: bool = False        # can it change state anywhere?
    reversible: bool = True     # can the change be undone cheaply?
    scope: str = "self"         # self | project | tenant | org | internet

    @property
    def plane(self) -> str:
        # A tool that cannot write is a read tool no matter what it is called.
        return "action" if self.writes else "decision"

# A security finding-triage agent, as actually deployed in a lot of places.
manifest = [
    Tool("search_findings"),
    Tool("read_source"),
    Tool("post_jira_comment", writes=True, scope="project"),
    Tool("close_finding",     writes=True, scope="project"),
    Tool("open_pr",           writes=True, scope="project"),
    Tool("merge_pr",          writes=True, scope="project", reversible=False),
    Tool("rotate_credential", writes=True, scope="org", reversible=False),
]

print(f"{'tool':22s}{'plane':10s}{'scope':10s}reversible")
print("-" * 56)
for t in manifest:
    print(f"{t.name:22s}{t.plane:10s}{t.scope:10s}{t.reversible}")

reads  = [t.name for t in manifest if not t.writes]
writes = [t.name for t in manifest if t.writes]
print(f"\ndecision plane ({len(reads)}): {reads}")
print(f"action plane   ({len(writes)}): {writes}")
print("\nThe model can propose all seven. Only the five on the action plane")
print("can change anything, and only if the control plane lets them through.")

## 3 · Where it breaks

Look at that manifest again. There is **no control plane in it at all** — nothing between the model's proposal and the tool call. Whatever the model emits, happens.

That is not a hypothetical configuration. It is the default: you give a framework a list of tools, and it calls them. The trust boundary that used to exist between "a human decided" and "the system executed" is gone, and nothing in the code review shows it missing, because *no code was written to remove it*.

So the first architecture question for an agent is not "is the code safe?" It is: **what is the worst single call this manifest permits, and who reviewed that?**

In [ ]:
def blast_radius(tools, gated=frozenset()):
    """What one unreviewed action can cost. Gated calls score zero — they are
    reviewed by definition."""
    per, total = {}, 0
    for t in tools:
        if not t.writes:
            continue
        score = SCOPE_WEIGHT[t.scope]
        if not t.reversible:
            score *= 2          # you cannot review an action after undoing it
        if t.name in gated:
            score = 0
        per[t.name] = score
        total += score
    return total, dict(sorted(per.items(), key=lambda kv: -kv[1]))

total, per = blast_radius(manifest)
print("ungoverned manifest — blast radius:", total)
for name, score in per.items():
    print(f"   {name:22s}{score:4d}")
print("\nThe worst single call is rotate_credential: org-wide and irreversible.")
print("An agent that can close a finding can also rotate the credential that")
print("finding was about. Nobody decided that; it fell out of the tool list.")

## 4 · The control

The fix is not to remove tools — the agent needs them to be useful. It is to put a **control plane** in the path, and the cheapest one that works is an approval gate on the calls that are wide or irreversible.

This is the first appearance of the **autonomy ladder**, which the rest of the curriculum uses constantly:

| Rung | What it means |
|---|---|
| **L1** | Model proposes, a human performs every action. |
| **L2** | Model calls tools, a human approves each call. |
| **L2.5** | Pre-approved tool set, bounded scope, humans review after the fact. |
| **L3** | Model acts and self-verifies; humans see aggregates. |

The rung is **not** about how clever the model is. It is about what the model's output is allowed to trigger without a human in the path. A small local model at L3 is more dangerous than a frontier model at L1.

In [ ]:
LADDER = {"L1": 0, "L2": 1, "L2.5": 3, "L3": 5}

def review(tools, gated, claimed_rung):
    """The architecture review, as a function. This is the deliverable."""
    problems = []
    writers = [t for t in tools if t.writes]
    ungated = [t.name for t in writers if t.name not in gated]
    if claimed_rung == "L1" and writers:
        problems.append(f"claims L1 but holds {len(writers)} state-changing tools")
    if claimed_rung == "L2" and ungated:
        problems.append(f"claims L2 (approve every call) but ungated: {ungated}")
    if claimed_rung == "L2.5":
        wide = [t.name for t in writers
                if SCOPE_WEIGHT[t.scope] >= SCOPE_WEIGHT["org"] and t.name not in gated]
        if wide:
            problems.append(f"claims L2.5 (bounded) but reaches org-wide ungated: {wide}")
    irrev = [t.name for t in writers if not t.reversible and t.name not in gated]
    if irrev and claimed_rung != "L3":
        problems.append(f"irreversible and ungated (no after-the-fact review possible): {irrev}")
    return problems

for label, gated, rung in [
    ("as deployed",                set(),                                      "L2.5"),
    ("gate the irreversible ones", {"merge_pr", "rotate_credential"},          "L2.5"),
    ("gate every writer",          {t.name for t in manifest if t.writes},     "L2"),
]:
    total, _ = blast_radius(manifest, gated)
    problems = review(manifest, gated, rung)
    print(f"{label:30s} rung={rung:5s} blast={total:3d}  "
          f"{'CLEAN' if not problems else 'FINDINGS'}")
    for p in problems:
        print(f"{'':32s}⚠ {p}")

## 5 · Verify — the review that runs itself

A threat model written in a document is stale the moment someone adds a tool, and adding a tool is a config change that no code review sees. So the artefact that actually protects you is not the model — it is the **diff**, run automatically whenever the manifest changes.

In [ ]:
def diff(before, after, gated=frozenset()):
    b, a = {t.name for t in before}, {t.name for t in after}
    tb, _ = blast_radius(before, gated)
    ta, _ = blast_radius(after, gated)
    return {"added": sorted(a - b), "removed": sorted(b - a),
            "blast": f"{tb} → {ta}", "delta": ta - tb,
            "new_findings": [p for p in review(after, gated, "L2.5")
                             if p not in review(before, gated, "L2.5")]}

gated = {"merge_pr", "rotate_credential"}
v1 = [t for t in manifest if t.name != "rotate_credential"]
v2 = manifest                               # someone adds credential rotation

print("someone edits the manifest on a Tuesday:")
for k, v in diff(v1, v2, gated={"merge_pr"}).items():
    print(f"   {k:14s} {v}")
print("\nNo pull request touched the agent's code. The blast radius doubled.")
print("This diff, wired into CI, IS the living architecture review.")

## What you just proved

The manifest splits into 2 decision-plane and 5 action-plane tools. Ungoverned it scores a blast radius of 55, with `rotate_credential` (40) dominating. Gating the two irreversible tools cuts it to 9 and clears every finding. The manifest diff shows `rotate_credential` being added, the blast radius going 15 → 55, and a new finding for an irreversible ungated tool.

## Your turn

Write out the manifest of one agent that is running in your organisation right now — every tool, honestly, including the ones added after launch. Run `review()` against the rung your team claims. The usual result is that the claimed rung is one or two above what the controls support.

---

**Next → [A1.4 · Designing the agent control plane](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*